# QC-VORONOI — Controle de Qualidade de Dados Pluviométricos

**Pipeline completo de análise:**
1. Carregamento dos dados brutos do HDF5
2. Exploração inicial (mapa de estações, estatísticas básicas)
3. Detecção de eventos chuvosos (MITs: 30 min, 6h, 24h)
4. Controle de qualidade temporal (dias válidos)
5. Controle de qualidade espacial (Triangulação de Delaunay)
6. Fine tuning dos parâmetros p1/p2
7. Geração de figuras e exportação de resultados

> Cada resultado intermediário é arquivado com timestamp em `archive/`
> e registrado em `src/pipeline.log`

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')

from src.config import ROOT, HDF5_PATH, UNIPLU_DIR, RESULTS_DIR, FIGURES_DIR, MITS_MINUTES
from src.data import (
    load_gauge_info, load_rainfall_data, load_daily_data,
    get_gauge_data_by_code, get_stations_by_network
)
from src.events import (
    determine_rainfall_events, compute_event_statistics,
    count_valid_days, create_days_record, compute_rainfall_time_step
)
from src.qc import (temporal_consistency_check, run_qc_delaunay,
    fine_tune_p1_p2, compute_confusion_matrix,
    bootstrap_all_metrics, mcnemar_test, temporal_cross_validation,
    temporal_cv_fine_tune, permutation_test, sensitivity_surface, find_plateau_region)
from src.visualization import (
    plot_station_map, plot_annual_rainfall_histogram,
    plot_event_properties_boxplot, plot_mit_comparison,
    plot_qc_f1_heatmap, plot_confusion_matrix, plot_f1_curve
)
from src.utils import save_result, save_figure, clear_log

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
clear_log()

print(f'UNIPLU: {UNIPLU_DIR}')
print(f'Resultados: {RESULTS_DIR}')
print(f'Figuras: {FIGURES_DIR}')
print(f'Log: {ROOT / "src" / "pipeline.log"}')

---
## 1. Carregamento dos Dados

In [ ]:
print('Carregando info das estações...')
df_info = load_gauge_info()
print(f'{len(df_info)} estações cadastradas')
print(f'Redes: {df_info["network"].value_counts().to_dict()}')
print(f'Estados: {sorted(df_info["state"].unique())}')
df_info.head()

In [ ]:
print('Carregando dados diários (resumo)...')
df_daily = load_daily_data()
print(f'Registros diários: {len(df_daily):,}')
print(f'Período: {df_daily["date"].min()} a {df_daily["date"].max()}')
df_daily.head()

### 1.1 Mapa das Estações

In [ ]:
fig = plot_station_map(df_info, save=False)
save_figure(fig, 'station_map_all.png')
plt.show()

In [ ]:
fig = plot_station_map(df_info, quality_labels=None, save=False)
ax = fig.axes[0]
for net, color in zip(['Telemetria', 'CEMADEN'], ['blue', 'orange']):
    subset = df_info[df_info['network'] == net]
    ax.scatter(subset['long'], subset['lat'], c=color, s=8, alpha=0.5,
               label=net, edgecolors='black', linewidth=0.2)
ax.legend(markerscale=3)
ax.set_title('Stations by Network')
save_figure(fig, 'station_map_by_network.png')
plt.show()

---
## 2. Carregamento dos Dados Brutos (Amostra)

> O dataset completo tem ~170M registros. Vamos trabalhar com uma **amostra**.
> Aumente `N_SAMPLE_STATIONS` para processar mais estações.

In [ ]:
N_SAMPLE_STATIONS = 50
SAMPLE_YEARS = [2023, 2024]
df_quality = None  # preenchido se houver ground truth

print(f'Amostrando {N_SAMPLE_STATIONS} estações ({SAMPLE_YEARS})...')
df_rainfall = load_rainfall_data(sample=N_SAMPLE_STATIONS, years=SAMPLE_YEARS, seed=42)
print(f'Registros carregados: {len(df_rainfall):,}')
print(f'Estações: {df_rainfall["gauge_code"].nunique()}')
print(f'Anos: {sorted(df_rainfall["datetime"].dt.year.unique())}')
df_rainfall.head()


---
## 3. Detecção de Eventos Chuvosos

Para cada estação e cada MIT, identificamos eventos chuvosos e calculamos:
- Lâmina precipitada (mm)
- Duração (horas)
- Intensidade média (mm/h)
- Tempo seco (horas)

In [ ]:
all_events = []
valid_days_records = []
codes_sample = df_rainfall['gauge_code'].unique()

for code in codes_sample:
    data = get_gauge_data_by_code(df_rainfall, code)
    if len(data) < 2:
        continue
    time_step = compute_rainfall_time_step(data)
    info_row = df_info[df_info['gauge_code'] == code]
    if info_row.empty:
        continue
    info_row = info_row.iloc[0]

    for year, yearly_data in data.groupby(data['datetime'].dt.year):
        n_days = count_valid_days(yearly_data)
        vd = create_days_record(year, code, n_days)
        valid_days_records.append(vd)

        rain_pos = yearly_data[yearly_data['rain_mm'] > 0].copy()
        rain_pos.reset_index(drop=True, inplace=True)
        if len(rain_pos) < 2:
            continue

        for mit_min in MITS_MINUTES:
            mit = pd.Timedelta(mit_min, unit='m')
            events_df = determine_rainfall_events(mit, rain_pos, time_step)
            if events_df.empty:
                continue
            stats = compute_event_statistics(events_df, info_row, mit_min)
            if stats:
                stats['year'] = year
                stats['n_valid_days'] = n_days
                all_events.append(stats)

df_events = pd.DataFrame(all_events)
df_valid = pd.concat(valid_days_records, ignore_index=True)

print(f'Eventos detectados: {len(df_events)}')
print(f'Estações processadas: {df_events["gauge_code"].nunique()}')
print(f'MITs: {sorted(df_events["mit_minutes"].unique())}')
df_events.head()


In [ ]:
print('=== Dias válidos por estação ===')
print(df_valid.describe())

In [ ]:
# Salva com log + timestamp archive
save_result(df_events, 'rainfall_events_sample.csv')
save_result(df_valid, 'valid_days_sample.csv')

---
## 4. Visualização das Propriedades dos Eventos

In [ ]:
for mit in MITS_MINUTES:
    fig = plot_annual_rainfall_histogram(df_events, mit_minutes=mit, save=False)
    save_figure(fig, f'annual_rainfall_hist_mit{mit}.png')
    plt.show()
    plt.close(fig)

In [ ]:
for mit in MITS_MINUTES:
    fig = plot_event_properties_boxplot(df_events, mit_minutes=mit, save=False)
    save_figure(fig, f'event_props_boxplot_mit{mit}.png')
    plt.show()
    plt.close(fig)

In [ ]:
if len(df_events['mit_minutes'].unique()) > 1:
    fig = plot_mit_comparison(df_events, save=False)
    save_figure(fig, 'mit_comparison.png')
    plt.show()
    plt.close(fig)

---
## 5. Controle de Qualidade Temporal

In [ ]:
codes = df_events['gauge_code'].unique()
tc_results = []
for code in codes:
    res = temporal_consistency_check(df_daily, code)
    tc_results.append(res)
df_tc = pd.DataFrame(tc_results)
save_result(df_tc, 'temporal_qc_results.csv')

print(f'Estações avaliadas: {len(df_tc)}')
print(f'Válidas: {df_tc["valid"].sum()}  |  Inválidas: {(~df_tc["valid"]).sum()}')
df_tc.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_tc['valid_days'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(300, color='red', linestyle='--', label='Limiar 300 dias')
axes[0].set_xlabel('Dias Válidos'); axes[0].set_ylabel('Frequência')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(df_tc['total_rainfall_mm'], bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(300, color='red', linestyle='--', label='Limiar 300 mm')
axes[1].set_xlabel('Precipitação Anual (mm)'); axes[1].set_ylabel('Frequência')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].hist(df_tc['max_gap_days'].clip(0, 60), bins=30, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Maior Falha (dias, limitado a 60)')
axes[2].set_ylabel('Frequência'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, 'temporal_qc_histograms.png')
plt.show()

---
## 6. QC Espacial — Triangulação de Delaunay

Para cada estação alvo, encontramos vizinhos via Delaunay entre as estações de referência.
Comparamos as propriedades: se o valor estiver dentro de `[média * p2, média * p1]`, é **HQ**; senão, **LQ**.

### 6.1 Fine Tuning de p1 e p2

In [ ]:
TARGET_YEAR = 2023
REFERENCE_YEARS = [2022, 2021, 2020]

print(f'Fine tuning: target={TARGET_YEAR}, reference={REFERENCE_YEARS}')

df_events['mit_minutes'] = df_events['mit_minutes'].astype(int)

qc_results_by_mit = {}
for mit in MITS_MINUTES:
    print(f'\n--- MIT={mit} min ---')
    df_mit = df_events[df_events['mit_minutes'] == mit].copy()
    if len(df_mit) < 5:
        print(f'Poucos dados, pulando...')
        continue

    results = fine_tune_p1_p2(
        df_events=df_mit, df_quality=pd.DataFrame(),
        gauge_info=df_info, mit_minutes=mit,
        target_year=TARGET_YEAR, reference_years=REFERENCE_YEARS
    )
    if not results.empty:
        qc_results_by_mit[mit] = results
        best = results.loc[results['f1'].idxmax()]
        print(f'Melhor: p1={best["p1"]:.2f}, p2={best["p2"]:.2f}, '
              f'F1={best["f1"]:.3f}, Acc={best["accuracy"]:.3f}')
        save_result(results, f'fine_tuning_mit{mit}.csv')

### 6.2 Heatmap e Curvas de F1

In [ ]:
for mit, df_res in qc_results_by_mit.items():
    fig = plot_qc_f1_heatmap(df_res, mit_minutes=mit, save=False)
    save_figure(fig, f'f1_heatmap_mit{mit}.png')
    plt.show()
    plt.close(fig)

In [ ]:
from src.config import P1_VALUES as p1_vals, P2_VALUES as p2_vals

for mit, df_res in qc_results_by_mit.items():
    fig = plot_f1_curve(df_res, p1_values=p1_vals, p2_values=p2_vals,
                          mit=mit, save=False)
    save_figure(fig, f'f1_curve_mit{mit}.png')
    plt.show()
    plt.close(fig)

### 6.3 Matriz de Confusão

In [ ]:
for mit, df_res in qc_results_by_mit.items():
    if df_res.empty:
        continue
    best = df_res.loc[df_res['f1'].idxmax()]
    cm_vals = (int(best['tp']), int(best['fp']),
               int(best['tn']), int(best['fn']))
    fig = plot_confusion_matrix(cm_vals, save=False,
        title=f'MIT={mit}min (p1={best["p1"]:.2f}, p2={best["p2"]:.2f})')
    save_figure(fig, f'confusion_matrix_mit{mit}.png')
    plt.show()
    plt.close(fig)

### 6.4 Classificação Final

In [ ]:
BEST_P1 = 1.60
BEST_P2 = 0.70

print(f'Classificando com p1={BEST_P1}, p2={BEST_P2}')

final_qc = []
for mit in MITS_MINUTES:
    df_mit = df_events[df_events['mit_minutes'] == mit].copy()
    if df_mit.empty:
        continue
    ref_df = df_mit[df_mit['year'] != TARGET_YEAR] if 'year' in df_mit.columns else df_mit
    tgt_df = df_mit[df_mit['year'] == TARGET_YEAR] if 'year' in df_mit.columns else df_mit

    for _, row in tgt_df.iterrows():
        qc = run_qc_delaunay(row, ref_df, TARGET_YEAR, mit,
                             BEST_P1, BEST_P2, df_info)
        if qc:
            row_dict = row.to_dict()
            row_dict.update(qc)
            final_qc.append(row_dict)

df_final_qc = pd.DataFrame(final_qc)
if not df_final_qc.empty:
    save_result(df_final_qc, 'final_qc_classification.csv')
    print(f'{len(df_final_qc)} classificações')
    print(df_final_qc['hybrid_quality'].value_counts())
    df_final_qc.head()

---
## 7. Mapa de Qualidade

In [ ]:
if not df_final_qc.empty:
    quality_map = df_final_qc[['gauge_code', 'hybrid_quality']].drop_duplicates()
    quality_map.columns = ['gauge_code', 'quality']
    fig = plot_station_map(df_info, quality_labels=quality_map, save=False)
    save_figure(fig, 'station_map_quality.png')
    plt.show()
    plt.close(fig)

---
## 8. Resumo Final

In [ ]:
from src.utils import read_log

print('='*60)
print('RESUMO DO PIPELINE QC-VORONOI')
print('='*60)
print(f'Total de estações: {len(df_info)}')
print(f'Estações analisadas: {df_events["gauge_code"].nunique()}')
print(f'Eventos chuvosos: {len(df_events)}')

n_results = len(list(RESULTS_DIR.glob('*.csv')))
n_figures = len(list(FIGURES_DIR.glob('*.png')))
n_archived = len(list((RESULTS_DIR/'archive').glob('*'))) + len(list((FIGURES_DIR/'archive').glob('*')))
print(f'\nResultados: {n_results} CSV  |  Figuras: {n_figures} PNG  |  Arquivados: {n_archived}')

print(f'\n=== Últimas entradas do log ===')
print(read_log(tail=10))

---
## 9. Validação Estatística dos Resultados

> Métodos aplicados:
> 1. **Bootstrap IC 95%** — estabilidade das métricas
> 2. **McNemar** — diferença entre MITs é significativa?
> 3. **Validação Cruzada Temporal** — modelo funciona em outros anos?
> 4. **Teste de Permutação** — modelo é melhor que aleatório?
> 5. **Superfície de Sensibilidade** — platô de p1/p2

### 9.1 Bootstrap — Intervalos de Confiança

In [ ]:
# ⚠️ Bootstrap requer ground truth real
# Se df_quality tiver rótulos reais (HQ/LQ de fonte externa),
# as células 9.1–9.5 produzem resultados válidos.
# Caso contrário, bootstrap com auto-validação é CIRCULAR (F1≡1.0) e foi removido.
#
# Para ativar: carregue df_quality real e descomente o bloco abaixo.

from collections import defaultdict

    has_real_quality = isinstance(df_quality, pd.DataFrame) and not df_quality.empty
if has_real_quality:
    predictions_by_mit = defaultdict(lambda: {'pred': [], 'true': []})
    for mit, df_grid in qc_results_by_mit.items():
        if df_grid.empty:
            continue
        best = df_grid.loc[df_grid['f1'].idxmax()]
        df_mit = df_events[df_events['mit_minutes'] == mit]
        ref_df = df_mit[df_mit['year'] != TARGET_YEAR]
        tgt_df = df_mit[df_mit['year'] == TARGET_YEAR]
        for _, row in tgt_df.iterrows():
            qc = run_qc_delaunay(row, ref_df, TARGET_YEAR, mit,
                                 best['p1'], best['p2'], df_info)
            if qc:
                predictions_by_mit[mit]['pred'].append(qc['hybrid_quality'])
                code = row['gauge_code']
                gt = df_quality[df_quality['gauge_code'] == code]
                if not gt.empty:
                    predictions_by_mit[mit]['true'].append(gt.iloc[0]['quality'])
        print(f'MIT={mit}min: {len(predictions_by_mit[mit]["pred"])} predições coletadas')
    print()
    for mit in sorted(predictions_by_mit.keys()):
        d = predictions_by_mit[mit]
        if len(d['pred']) < 10:
            continue
        y_t = np.array(d['true'])
        y_p = np.array(d['pred'])
        ci = bootstrap_all_metrics(y_t, y_p, n_iter=2000, ci=0.95)
        print(f'--- MIT={mit} min ---')
        for metric, vals in ci.items():
            print(f'  {metric:>12}: {vals["mean"]:.4f}  '
                  f'IC95% [{vals["lower"]:.4f}, {vals["upper"]:.4f}]  '
                  f'std={vals["std"]:.4f}')
else:
    print('Ground truth (df_quality) não disponível.')
    print('Bootstrap, McNemar, permutação e CV temporal serão SKIPPADOS.')
    print('Carregue df_quality com rótulos reais e reexecute esta seção.')

### 9.2 Teste de McNemar — MIT 30min vs MIT 6h vs MIT 24h

In [ ]:
# Para McNemar precisamos de predições pareadas dos dois modelos no MESMO conjunto
# Usamos as classificações do fine tuning com os melhores (p1,p2) de cada MIT
mits = sorted(predictions_by_mit.keys())
if len(mits) >= 2:
    print('Comparações pareadas entre MITs (McNemar):')
    for i in range(len(mits)):
        for j in range(i+1, len(mits)):
            mit_a, mit_b = mits[i], mits[j]
            pred_a = predictions_by_mit[mit_a]['pred']
            pred_b = predictions_by_mit[mit_b]['pred']
            true = predictions_by_mit[mit_a]['true']
            # Apenas estações com predição em ambos
            min_len = min(len(pred_a), len(pred_b))
            if min_len < 10:
                continue
            result = mcnemar_test(np.array(true[:min_len]),
                                  np.array(pred_a[:min_len]),
                                  np.array(pred_b[:min_len]))
            sig = 'SIGNIFICATIVO' if result['significant'] else 'não significativo'
            print(f'  MIT{mit_a} vs MIT{mit_b}: '
                  f'chi2={result["chi2"]:.3f}, p={result["p_value"]:.4f} ({sig})')
else:
    print('Poucos MITs com dados para McNemar.')

### 9.3 Validação Cruzada Temporal

In [ ]:
if 'year' in df_events.columns and df_events['year'].nunique() >= 2:
    print('Validação cruzada temporal (cada ano como teste)...')
    for mit in MITS_MINUTES:
        df_mit = df_events[df_events['mit_minutes'] == mit].copy()
        if df_mit['year'].nunique() < 2:
            continue
        df_cv = temporal_cross_validation(
            df_mit, pd.DataFrame(), df_info, mit,
            years=sorted(df_mit['year'].unique()), p1=1.6, p2=0.8
        )
        if len(df_cv) > 1:
            overall = df_cv.iloc[-1]
            print(f'  MIT={mit}min: F1={overall["f1"]:.3f} ± {overall.get("f1_std",0):.3f}, '
                  f'Acc={overall["accuracy"]:.3f} ± {overall.get("accuracy_std",0):.3f}')
            save_result(df_cv, f'temporal_cv_mit{mit}.csv')
        else:
            print(f'  MIT={mit}min: dados insuficientes')
else:
    print('Apenas 1 ano disponível — CV temporal não aplicável.')

### 9.4 Teste de Permutação — modelo é melhor que aleatório?

In [ ]:
print('Teste de permutação (5000 iterações)...')
for mit in sorted(predictions_by_mit.keys()):
    d = predictions_by_mit[mit]
    if len(d['pred']) < 10:
        continue
    result = permutation_test(
        np.array(d['true']), np.array(d['pred']),
        n_perm=2000
    )
    sig = 'SIGNIFICATIVO' if result['significant'] else 'não significativo'
    print(f'  MIT={mit}min: F1 real={result["real_score"]:.4f}, '
          f'perm_mean={result["perm_mean"]:.4f}, '
          f'p={result["p_value"]:.4f} ({sig})')

### 9.5 Superfície de Sensibilidade — Platô de p1/p2

In [ ]:
from src.config import P1_VALUES as p1_vals, P2_VALUES as p2_vals

for mit in MITS_MINUTES:
    df_mit = df_events[df_events['mit_minutes'] == mit].copy()
    if len(df_mit) < 5:
        continue
    surface = sensitivity_surface(
        df_mit, pd.DataFrame(), df_info, mit,
        TARGET_YEAR, REFERENCE_YEARS,
        p1_values=p1_vals, p2_values=p2_vals
    )
    if surface.empty:
        continue
    
    # Find plateau (pairs within 0.01 of best F1)
    plateau = find_plateau_region(surface, metric='f1', tolerance=0.01)
    print(f'\nMIT={mit}min — Melhores {len(plateau)} combinações (F1 a {0.01} do máximo):')
    for _, row in plateau.iterrows():
        print(f'  p1={row["p1"]:.2f}, p2={row["p2"]:.2f}: '
              f'F1={row["f1"]:.4f}, MCC={row["mcc"]:.4f}, '
              f'Youden={row["youden_index"]:.4f}')
    
    save_result(surface, f'sensitivity_surface_mit{mit}.csv')

### 9.6 Resumo da Validação

In [ ]:
print('='*70)
print('RESUMO DA VALIDAÇÃO ESTATÍSTICA')
print('='*70)
print()
print('1. Bootstrap IC 95%:')
print('   - F1, precisão, recall, acurácia com intervalos de confiança')
print('   - Ver resultados acima para cada MIT')
print()
print('2. McNemar:')
print('   - Testa se a diferença entre MITs é estatisticamente significativa')
print('   - p < 0.05 → os MITs têm desempenho diferente')
print()
print('3. Validação Cruzada Temporal:')
print('   - Cada ano vira teste, os demais são referência')
print('   - Reporta média e desvio do F1 entre os folds')
print()
print('4. Teste de Permutação:')
print('   - Embaralha os rótulos verdadeiros N vezes')
print('   - p < 0.05 → modelo é melhor que classificação aleatória')
print()
print('5. Superfície de Sensibilidade:')
print('   - MCC, Youden Index, balanced accuracy por (p1,p2)')
print('   - Platô: quantos pares (p1,p2) têm F1 próximo do máximo?')